# Iris Classification: Logistic Regression vs Decision Tree

**Objective:** Train and compare two supervised classification models using the Iris dataset.

This notebook:
- Loads the Iris CSV dataset with Pandas
- Performs exploratory analysis and pairplots
- Splits data into 80% training and 20% testing sets
- Scales features using `StandardScaler`
- Trains Logistic Regression and Decision Tree classifiers
- Reports accuracy, precision, recall, and F1-score
- Plots confusion-matrix heatmaps
- Visualizes decision boundaries using two features

> **Note:** Scaling is fitted only on the training data to prevent data leakage. The decision-boundary plots use two selected features for visualization; the benchmark models use all four features.

In [ ]:
# If needed, install dependencies in your notebook environment:
# %pip install pandas scikit-learn seaborn matplotlib

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

## 1. Load the dataset

The CSV is downloaded from the supplied Seaborn dataset URL. If internet access is unavailable, download `iris.csv` first and update `DATA_PATH` to its local path.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv"
DATA_PATH = DATA_URL  # Change to "iris.csv" if using a local copy

df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
print("Dataset shape:", df.shape)
print("\\nColumn names:", df.columns.tolist())
print("\\nMissing values per column:\\n", df.isnull().sum())
print("\\nClass counts:\\n", df["species"].value_counts())

In [ ]:
df.describe(include="all")

## 2. Exploratory Data Analysis (EDA)

The pairplot helps show how the four measurements vary across Iris species.

In [ ]:
sns.pairplot(df, hue="species", diag_kind="hist")
plt.suptitle("Iris Dataset Pairplot", y=1.02)
plt.show()

## 3. Prepare features and split the data

The target is `species`; the four numeric measurements are the features. The split is stratified so each species is represented proportionally in both sets.

In [ ]:
feature_cols = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
target_col = "species"

X = df[feature_cols]
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f"Training samples: {len(X_train)} ({len(X_train)/len(df):.0%})")
print(f"Testing samples:  {len(X_test)} ({len(X_test)/len(df):.0%})")

## 4. Train the models

Logistic Regression benefits from standardized features. The Decision Tree does not require scaling, but using a pipeline keeps the workflow consistent and lets us compare both models on the same split. The tree pipeline also scales features, though tree split decisions are generally unaffected by monotonic scaling.

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    ]),
    "Decision Tree": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", DecisionTreeClassifier(random_state=RANDOM_STATE))
    ])
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"{name}: training complete")

## 5. Evaluate and benchmark performance

Precision, recall, and F1-score use **macro averaging**, which calculates each species' score separately and then averages the results equally.

In [ ]:
results = []
predictions = {}

for name, model in models.items():
    y_pred = model.predict(X_test)
    predictions[name] = y_pred

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision (macro)": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "Recall (macro)": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "F1-score (macro)": f1_score(y_test, y_pred, average="macro", zero_division=0)
    })

results_df = pd.DataFrame(results).set_index("Model")
display(results_df.round(4))

In [ ]:
for name, y_pred in predictions.items():
    print(f"\\n===== {name} =====")
    print(classification_report(y_test, y_pred, zero_division=0))

## 6. Confusion matrix heatmaps

Rows represent the true species; columns represent the model's predicted species. Correct predictions appear along the main diagonal.

In [ ]:
class_labels = sorted(y.unique())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (name, y_pred) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, y_pred, labels=class_labels)
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues", cbar=False,
        xticklabels=class_labels, yticklabels=class_labels, ax=ax
    )
    ax.set_title(f"Confusion Matrix — {name}")
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
plt.tight_layout()
plt.show()

## 7. Compare metrics visually

In [ ]:
results_df.plot(kind="bar", figsize=(11, 6), ylim=(0, 1.05))
plt.title("Classifier Performance Comparison")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 8. Decision-boundary visualization

A 2D plot can show only two features. Here we use `petal_length` and `petal_width`. Each model is retrained using only those two features for this visualization; these are separate visualization models and are not the four-feature models used in the benchmark above.

In [ ]:
viz_features = ["petal_length", "petal_width"]
X2 = df[viz_features]

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

viz_models = {
    "Logistic Regression (2 features)": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    ]),
    "Decision Tree (2 features)": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", DecisionTreeClassifier(random_state=RANDOM_STATE))
    ])
}

for model in viz_models.values():
    model.fit(X2_train, y2_train)

x_min, x_max = X2[viz_features[0]].min() - 0.5, X2[viz_features[0]].max() + 0.5
y_min, y_max = X2[viz_features[1]].min() - 0.5, X2[viz_features[1]].max() + 0.5
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 300),
    np.linspace(y_min, y_max, 300)
)
grid = pd.DataFrame({
    viz_features[0]: xx.ravel(),
    viz_features[1]: yy.ravel()
})

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, (name, model) in zip(axes, viz_models.items()):
    grid_pred = model.predict(grid)
    # Map classes to integers for a colored region plot.
    label_to_num = {label: i for i, label in enumerate(class_labels)}
    zz = np.array([label_to_num[label] for label in grid_pred]).reshape(xx.shape)
    ax.contourf(xx, yy, zz, alpha=0.25, levels=np.arange(-0.5, len(class_labels) + 0.5, 1), cmap="viridis")
    sns.scatterplot(
        data=df, x=viz_features[0], y=viz_features[1],
        hue="species", palette="viridis", edgecolor="black",
        s=55, ax=ax, legend=(ax is axes[1])
    )
    ax.set_title(name)
    ax.set_xlabel(viz_features[0].replace("_", " ").title())
    ax.set_ylabel(viz_features[1].replace("_", " ").title())
plt.tight_layout()
plt.show()

## 9. Conclusion

Review the metric table and confusion matrices to decide which model performed better on this particular test split. A higher score is not a guarantee of better performance on every future dataset. The Iris dataset is small, so results can vary with the split and random state.

**Checklist**
- [x] Pandas dataset loading
- [x] Exploratory pairplot
- [x] 80/20 train-test split
- [x] StandardScaler
- [x] Logistic Regression and Decision Tree
- [x] Accuracy, precision, recall, and F1-score
- [x] Classification reports
- [x] Confusion matrix heatmaps
- [x] Decision-boundary visualization